# 面试问题：T5 的 span corruption 怎样构造，并如何手写一个可训练的 Encoder-Decoder？

## 可直接复述的回答主线

1. T5 不是逐 token 随机替换，而是把连续被遮蔽片段压缩成唯一 sentinel，输入更短且目标明确。
2. 目标序列按原文顺序写成 sentinel 加被删片段，最后追加 EOS；同一条样本的不同 span 不能复用 sentinel。
3. Decoder 训练采用 teacher forcing：输入是 BOS 加右移目标，标签仍是原目标，并用 causal mask 禁止偷看未来。
4. Encoder self-attention 双向，Decoder 同时包含 causal self-attention 与对 Encoder memory 的 cross-attention。
5. T5 常用相对位置偏置、RMSNorm 与共享词表；本例从底层写出这些核心部件并绑定输出 embedding。
6. 评测不能只看 loss，要和同数据 baseline 比 token accuracy，并展示逐样本生成、attention、logits 与错误 span。
7. 生产还要处理动态噪声、sentinel 预算、packing、长文截断、分布式训练、解码约束和数据污染。

后续实验会在同一批可读输入上依次展示基线、手写核心机制、训练过程、逐样本结果、失败修正与生产边界。

## 1. 真实案例与输入预览

案例包含 6 条可读的中文商品搜索句，每条指定一到两个连续 span。实验会逐条打印原文、被遮蔽位置、压缩后的 Encoder 输入和 Decoder 目标；随后用同一批样本比较位置众数 baseline 与手写 TinyT5。

In [1]:
import math  # 汇总梯度与注意力诊断指标。
import warnings  # 过滤本地 PyTorch 的无关兼容警告。
from collections import Counter  # 统计每个目标位置的 baseline 众数词。
warnings.filterwarnings("ignore", message=".*pynvml.*", category=FutureWarning)  # 保持输出聚焦序列建模过程。
import torch  # 使用基础张量和自动微分手写 Encoder-Decoder。
torch.manual_seed(431)  # 固定参数初始化与优化轨迹。
torch.set_num_threads(1)  # 固定 CPU 单线程以提高复现性。
raw_samples = [["用户", "搜索", "红色", "无线", "耳机"], ["客户", "查询", "北京", "门店", "库存"], ["推荐", "轻薄", "游戏", "笔记本", "电脑"], ["查找", "防水", "运动", "智能", "手表"], ["比较", "华为", "手机", "近期", "价格"], ["需要", "儿童", "学习", "平板", "电脑"]]  # 定义六条真实可读搜索文本。
sample_spans = [[(1, 2), (3, 4)], [(1, 3)], [(1, 2), (3, 4)], [(1, 3)], [(1, 2), (3, 4)], [(1, 3)]]  # 为每条文本指定不重叠的左闭右开连续 span。
special_tokens = ["<pad>", "<bos>", "<eos>", "<extra_id_0>", "<extra_id_1>", "<extra_id_2>"]  # 定义 padding、解码边界和唯一 sentinel。
ordinary_tokens = sorted({token for sample in raw_samples for token in sample})  # 收集案例中实际出现的中文词。
vocabulary = special_tokens + ordinary_tokens  # 构造共享 Encoder-Decoder 词表。
token_to_id = {token: index for index, token in enumerate(vocabulary)}  # 建立 token 到整数编号映射。
id_to_token = {index: token for token, index in token_to_id.items()}  # 建立整数编号到 token 映射。
pad_id = token_to_id["<pad>"]  # 保存 padding 编号。
bos_id = token_to_id["<bos>"]  # 保存 decoder 起始编号。
eos_id = token_to_id["<eos>"]  # 保存序列结束编号。
def span_corrupt(tokens, spans):  # 把连续遮蔽片段变成 T5 source 和 target。
    source_tokens = []  # 保存压缩后的 Encoder 输入。
    target_tokens = []  # 保存 sentinel 加被删内容的监督目标。
    span_by_start = {start: (end, index) for index, (start, end) in enumerate(spans)}  # 按起点索引每个遮蔽 span。
    position = 0  # 从原文首词开始扫描。
    while position < len(tokens):  # 逐位置复制明文或压缩 span。
        if position in span_by_start:  # 判断当前位置是否是一个遮蔽片段起点。
            end, sentinel_index = span_by_start[position]  # 读取片段终点和唯一 sentinel 序号。
            sentinel = f"<extra_id_{sentinel_index}>"  # 构造当前片段专属 sentinel。
            source_tokens.append(sentinel)  # 在 Encoder 输入中用一个 token 压缩整个片段。
            target_tokens.append(sentinel)  # 在目标中先声明接下来恢复哪个片段。
            target_tokens.extend(tokens[position:end])  # 按原顺序追加被删除的真实词。
            position = end  # 跳过已经压缩的原文片段。
        else:  # 处理未被遮蔽的普通位置。
            source_tokens.append(tokens[position])  # 把可见词原样复制到 Encoder 输入。
            position += 1  # 移到下一个原文位置。
    target_tokens.append("<eos>")  # 在恢复目标末尾追加停止符。
    return source_tokens, target_tokens  # 返回完整 source-target 对。
corrupted_pairs = [span_corrupt(tokens, spans) for tokens, spans in zip(raw_samples, sample_spans)]  # 为六条文本真正执行 span corruption。
for index, (tokens, spans, pair) in enumerate(zip(raw_samples, sample_spans, corrupted_pairs)):  # 逐条展示真实预训练输入。
    print(f"样本{index + 1} 原文={' '.join(tokens)}")  # 输出人类可读原始句子。
    print(f"       spans={spans} source={' '.join(pair[0])}")  # 输出被压缩的 Encoder 输入。
    print(f"       target={' '.join(pair[1])}")  # 输出 Decoder 应恢复的片段序列。

样本1 原文=用户 搜索 红色 无线 耳机
       spans=[(1, 2), (3, 4)] source=用户 <extra_id_0> 红色 <extra_id_1> 耳机
       target=<extra_id_0> 搜索 <extra_id_1> 无线 <eos>
样本2 原文=客户 查询 北京 门店 库存
       spans=[(1, 3)] source=客户 <extra_id_0> 门店 库存
       target=<extra_id_0> 查询 北京 <eos>
样本3 原文=推荐 轻薄 游戏 笔记本 电脑
       spans=[(1, 2), (3, 4)] source=推荐 <extra_id_0> 游戏 <extra_id_1> 电脑
       target=<extra_id_0> 轻薄 <extra_id_1> 笔记本 <eos>
样本4 原文=查找 防水 运动 智能 手表
       spans=[(1, 3)] source=查找 <extra_id_0> 智能 手表
       target=<extra_id_0> 防水 运动 <eos>
样本5 原文=比较 华为 手机 近期 价格
       spans=[(1, 2), (3, 4)] source=比较 <extra_id_0> 手机 <extra_id_1> 价格
       target=<extra_id_0> 华为 <extra_id_1> 近期 <eos>
样本6 原文=需要 儿童 学习 平板 电脑
       spans=[(1, 3)] source=需要 <extra_id_0> 平板 电脑
       target=<extra_id_0> 儿童 学习 <eos>


## 2. Baseline / 基线：每个目标位置都猜训练批次众数

这个 baseline 不读取 source，只在第 1、2、3…个目标位置分别猜六条样本中最常出现的 token。它与 TinyT5 使用完全相同的非 padding 目标位置和 token accuracy。

In [2]:
def encode(tokens):  # 把可读 token 列表转为整数编号。
    return [token_to_id[token] for token in tokens]  # 按共享词表逐词映射。
encoded_sources = [encode(pair[0]) for pair in corrupted_pairs]  # 编码六条 Encoder 输入。
encoded_targets = [encode(pair[1]) for pair in corrupted_pairs]  # 编码六条 Decoder 标签。
source_length = max(len(row) for row in encoded_sources)  # 找到 source 批次最大长度。
target_length = max(len(row) for row in encoded_targets)  # 找到 target 批次最大长度。
def pad_rows(rows, length):  # 把变长整数序列补齐为批张量。
    return torch.tensor([row + [pad_id] * (length - len(row)) for row in rows], dtype=torch.long)  # 在每行右侧追加 padding。
source_ids = pad_rows(encoded_sources, source_length)  # 构造六乘 source 长度输入张量。
target_ids = pad_rows(encoded_targets, target_length)  # 构造六乘 target 长度标签张量。
source_mask = source_ids.ne(pad_id)  # 标记 Encoder 中真实 token。
target_mask = target_ids.ne(pad_id)  # 标记损失和评测中的真实目标 token。
decoder_input_ids = torch.cat([torch.full((len(raw_samples), 1), bos_id, dtype=torch.long), target_ids[:, :-1]], dim=1)  # 用 BOS 加右移目标构造 teacher-forcing 输入。
decoder_mask = decoder_input_ids.ne(pad_id)  # 标记 Decoder 当前可见的真实输入位置。
majority_by_position = []  # 保存每个目标位置的众数 token。
for position in range(target_length):  # 逐目标位置建立无条件 baseline。
    valid_tokens = target_ids[target_mask[:, position], position].tolist()  # 读取该位置所有非 padding 标签。
    majority_by_position.append(Counter(valid_tokens).most_common(1)[0][0] if valid_tokens else eos_id)  # 选择频次最高 token。
baseline_predictions = torch.tensor(majority_by_position, dtype=torch.long).unsqueeze(0).repeat(len(raw_samples), 1)  # 对所有样本广播同一位置模板。
baseline_correct = baseline_predictions.eq(target_ids) & target_mask  # 标记 baseline 在有效目标上的命中。
baseline_accuracy = baseline_correct.sum().item() / target_mask.sum().item()  # 计算同口径 token accuracy。
def decode_ids(ids):  # 把整数序列还原为易读 token 字符串。
    return " ".join(id_to_token[int(index)] for index in ids if int(index) != pad_id)  # 跳过 padding 并连接真实 token。
print(f"source_ids shape={tuple(source_ids.shape)} target_ids shape={tuple(target_ids.shape)}")  # 展示真实 batch 张量形状。
print(f"Baseline token accuracy={baseline_accuracy:.3f}")  # 输出无条件位置众数基线。
for index in range(len(raw_samples)):  # 逐样本展示 baseline 的具体猜测。
    print(f"样本{index + 1} baseline={decode_ids(baseline_predictions[index])} target={decode_ids(target_ids[index])} token_acc={baseline_correct[index].sum().item() / target_mask[index].sum().item():.3f}")  # 展示预测、标签和逐样本准确率。

source_ids shape=(6, 5) target_ids shape=(6, 5)
Baseline token accuracy=0.593
样本1 baseline=<extra_id_0> 搜索 <extra_id_1> <eos> <eos> target=<extra_id_0> 搜索 <extra_id_1> 无线 <eos> token_acc=0.800
样本2 baseline=<extra_id_0> 搜索 <extra_id_1> <eos> <eos> target=<extra_id_0> 查询 北京 <eos> token_acc=0.500
样本3 baseline=<extra_id_0> 搜索 <extra_id_1> <eos> <eos> target=<extra_id_0> 轻薄 <extra_id_1> 笔记本 <eos> token_acc=0.600
样本4 baseline=<extra_id_0> 搜索 <extra_id_1> <eos> <eos> target=<extra_id_0> 防水 运动 <eos> token_acc=0.500
样本5 baseline=<extra_id_0> 搜索 <extra_id_1> <eos> <eos> target=<extra_id_0> 华为 <extra_id_1> 近期 <eos> token_acc=0.600
样本6 baseline=<extra_id_0> 搜索 <extra_id_1> <eos> <eos> target=<extra_id_0> 儿童 学习 <eos> token_acc=0.500


## 3. 底层实现：相对位置偏置、RMSNorm、三种 Attention 与共享词表

下面不调用现成 Transformer/T5。Encoder 使用双向 self-attention；Decoder 先做带下三角 mask 的 self-attention，再以 Decoder hidden 作 query、Encoder memory 作 key/value。输出层直接与共享 embedding 权重绑定。

In [3]:
class RMSNorm(torch.nn.Module):  # 实现 T5 风格均方根归一化。
    def __init__(self, dimension, epsilon=1.0e-6):  # 初始化可学习缩放参数。
        super().__init__()  # 注册 PyTorch 模块。
        self.scale = torch.nn.Parameter(torch.ones(dimension))  # 为每个 hidden 维度学习缩放。
        self.epsilon = epsilon  # 保存数值稳定常数。
    def forward(self, values):  # 对最后一维执行 RMSNorm。
        inverse_rms = torch.rsqrt(values.pow(2).mean(dim=-1, keepdim=True) + self.epsilon)  # 计算均方根倒数而不减均值。
        return values * inverse_rms * self.scale  # 返回归一化并缩放的 hidden。
class RelativePositionBias(torch.nn.Module):  # 实现可学习的裁剪相对距离偏置。
    def __init__(self, heads, max_distance=8):  # 初始化每个相对距离和注意力头的参数。
        super().__init__()  # 注册 PyTorch 模块。
        self.max_distance = max_distance  # 保存距离裁剪边界。
        self.embedding = torch.nn.Embedding(2 * max_distance + 1, heads)  # 为负到正相对距离学习每头偏置。
    def forward(self, query_length, key_length):  # 生成一张 query-key 相对位置表。
        query_positions = torch.arange(query_length)[:, None]  # 构造纵向 query 位置。
        key_positions = torch.arange(key_length)[None, :]  # 构造横向 key 位置。
        relative_positions = (key_positions - query_positions).clamp(-self.max_distance, self.max_distance)  # 计算并裁剪有符号相对距离。
        bias_indices = relative_positions + self.max_distance  # 平移为 embedding 的非负索引。
        return self.embedding(bias_indices).permute(2, 0, 1).unsqueeze(0)  # 返回一乘头乘 query 乘 key 偏置。
class ManualAttention(torch.nn.Module):  # 从线性投影手写多头注意力。
    def __init__(self, dimension=32, heads=4):  # 初始化 QKV 与输出投影。
        super().__init__()  # 注册所有可学习层。
        self.dimension = dimension  # 保存 hidden 总维度。
        self.heads = heads  # 保存注意力头数。
        self.head_dimension = dimension // heads  # 计算每头维度。
        self.query = torch.nn.Linear(dimension, dimension, bias=False)  # 创建 query 投影。
        self.key = torch.nn.Linear(dimension, dimension, bias=False)  # 创建 key 投影。
        self.value = torch.nn.Linear(dimension, dimension, bias=False)  # 创建 value 投影。
        self.output = torch.nn.Linear(dimension, dimension, bias=False)  # 创建拼接后输出投影。
    def split_heads(self, values):  # 把 hidden 总维度拆成多个头。
        batch, length, _ = values.shape  # 读取批量和序列长度。
        return values.view(batch, length, self.heads, self.head_dimension).transpose(1, 2)  # 返回批乘头乘长度乘头维度。
    def forward(self, query_input, key_value_input, key_mask, relative_bias, causal=False):  # 计算 self 或 cross attention。
        queries = self.split_heads(self.query(query_input))  # 投影并拆分 query 多头。
        keys = self.split_heads(self.key(key_value_input))  # 投影并拆分 key 多头。
        values = self.split_heads(self.value(key_value_input))  # 投影并拆分 value 多头。
        scores = queries @ keys.transpose(-2, -1) / math.sqrt(self.head_dimension)  # 计算缩放点积注意力分数。
        scores = scores + relative_bias  # 注入与内容无关的相对位置偏置。
        scores = scores.masked_fill(~key_mask[:, None, None, :], -1.0e4)  # 屏蔽 padding key。
        if causal:  # 判断是否需要 Decoder 因果约束。
            query_length = query_input.shape[1]  # 读取当前 Decoder query 长度。
            key_length = key_value_input.shape[1]  # 读取当前 Decoder key 长度。
            causal_mask = torch.arange(key_length)[None, :] <= torch.arange(query_length)[:, None]  # 构造只允许看当前位置及过去的下三角 mask。
            scores = scores.masked_fill(~causal_mask[None, None, :, :], -1.0e4)  # 把未来位置分数压成极小值。
        weights = torch.softmax(scores, dim=-1)  # 把每行分数归一化为注意力概率。
        context = weights @ values  # 按注意力概率聚合 value。
        merged = context.transpose(1, 2).contiguous().view(query_input.shape[0], query_input.shape[1], self.dimension)  # 拼回所有注意力头。
        return self.output(merged), weights  # 返回上下文表示与可解释权重。
class FeedForward(torch.nn.Module):  # 定义逐 token 非线性前馈网络。
    def __init__(self, dimension=32, hidden_dimension=64):  # 初始化升维和降维投影。
        super().__init__()  # 注册 PyTorch 模块。
        self.input = torch.nn.Linear(dimension, hidden_dimension, bias=False)  # 把 hidden 升维。
        self.output = torch.nn.Linear(hidden_dimension, dimension, bias=False)  # 把激活降回 residual 维度。
    def forward(self, values):  # 执行 T5 风格 gated-free 前馈变换。
        return self.output(torch.nn.functional.gelu(self.input(values)))  # 使用 GELU 产生非线性 token 表示。
class EncoderBlock(torch.nn.Module):  # 组合双向 self-attention 与前馈层。
    def __init__(self, dimension=32, heads=4):  # 初始化 Encoder 子层。
        super().__init__()  # 注册 PyTorch 模块。
        self.attention_norm = RMSNorm(dimension)  # 创建 self-attention 前归一化。
        self.attention = ManualAttention(dimension, heads)  # 创建双向多头 self-attention。
        self.feed_norm = RMSNorm(dimension)  # 创建前馈层前归一化。
        self.feed_forward = FeedForward(dimension, dimension * 2)  # 创建逐位置前馈网络。
    def forward(self, hidden, mask, bias):  # 编码一批 source token。
        attention_output, attention_weights = self.attention(self.attention_norm(hidden), self.attention_norm(hidden), mask, bias, causal=False)  # 让每个 source 位置双向读取所有有效 token。
        hidden = hidden + attention_output  # 应用第一条 residual connection。
        hidden = hidden + self.feed_forward(self.feed_norm(hidden))  # 应用前馈变换和第二条 residual connection。
        return hidden, attention_weights  # 返回 Encoder memory 与注意力权重。
class DecoderBlock(torch.nn.Module):  # 组合因果 self-attention、cross-attention 与前馈层。
    def __init__(self, dimension=32, heads=4):  # 初始化 Decoder 三个子层。
        super().__init__()  # 注册 PyTorch 模块。
        self.self_norm = RMSNorm(dimension)  # 创建因果 self-attention 前归一化。
        self.self_attention = ManualAttention(dimension, heads)  # 创建 Decoder self-attention。
        self.cross_norm = RMSNorm(dimension)  # 创建 cross-attention query 前归一化。
        self.cross_attention = ManualAttention(dimension, heads)  # 创建读取 Encoder memory 的注意力。
        self.feed_norm = RMSNorm(dimension)  # 创建前馈层前归一化。
        self.feed_forward = FeedForward(dimension, dimension * 2)  # 创建逐位置前馈网络。
    def forward(self, hidden, decoder_mask, memory, source_mask, self_bias, cross_bias):  # 解码一批右移目标。
        normalized = self.self_norm(hidden)  # 归一化 Decoder 当前 hidden。
        self_output, self_weights = self.self_attention(normalized, normalized, decoder_mask, self_bias, causal=True)  # 只聚合当前及历史目标位置。
        hidden = hidden + self_output  # 应用 causal self-attention residual。
        cross_output, cross_weights = self.cross_attention(self.cross_norm(hidden), memory, source_mask, cross_bias, causal=False)  # 让每个目标位置读取完整 source memory。
        hidden = hidden + cross_output  # 应用 cross-attention residual。
        hidden = hidden + self.feed_forward(self.feed_norm(hidden))  # 应用前馈变换和 residual。
        return hidden, self_weights, cross_weights  # 返回 Decoder hidden 和两类 attention。
class TinyT5(torch.nn.Module):  # 从底层部件组装可训练的简化 T5。
    def __init__(self, vocabulary_size, dimension=32, heads=4):  # 初始化共享词表与单层 Encoder-Decoder。
        super().__init__()  # 注册完整模型参数。
        self.shared_embedding = torch.nn.Embedding(vocabulary_size, dimension)  # 创建 Encoder、Decoder 和输出共享词表矩阵。
        self.encoder_bias = RelativePositionBias(heads)  # 创建 Encoder 相对位置偏置。
        self.decoder_bias = RelativePositionBias(heads)  # 创建 Decoder self-attention 相对位置偏置。
        self.cross_bias = RelativePositionBias(heads)  # 创建 Decoder-to-Encoder 相对位置偏置。
        self.encoder = EncoderBlock(dimension, heads)  # 创建一层 Encoder。
        self.decoder = DecoderBlock(dimension, heads)  # 创建一层 Decoder。
        self.final_norm = RMSNorm(dimension)  # 创建输出前归一化。
    def forward(self, source, source_valid, decoder_input, decoder_valid):  # 执行完整 Encoder-Decoder 前向传播。
        source_hidden = self.shared_embedding(source)  # 用共享矩阵嵌入 source token。
        encoder_bias = self.encoder_bias(source.shape[1], source.shape[1])  # 生成 source-to-source 相对偏置。
        memory, encoder_attention = self.encoder(source_hidden, source_valid, encoder_bias)  # 编码双向 source memory。
        decoder_hidden = self.shared_embedding(decoder_input)  # 用同一矩阵嵌入右移目标。
        decoder_bias = self.decoder_bias(decoder_input.shape[1], decoder_input.shape[1])  # 生成 target-to-target 相对偏置。
        cross_bias = self.cross_bias(decoder_input.shape[1], source.shape[1])  # 生成 target-to-source 相对偏置。
        decoder_hidden, decoder_attention, cross_attention = self.decoder(decoder_hidden, decoder_valid, memory, source_valid, decoder_bias, cross_bias)  # 运行三子层 Decoder。
        normalized = self.final_norm(decoder_hidden)  # 归一化最终 Decoder hidden。
        logits = torch.nn.functional.linear(normalized, self.shared_embedding.weight)  # 用共享 embedding 转置产生词表 logits。
        debug = {"memory": memory, "encoder_attention": encoder_attention, "decoder_attention": decoder_attention, "cross_attention": cross_attention, "encoder_bias": encoder_bias, "decoder_bias": decoder_bias, "decoder_hidden": normalized}  # 保存关键中间张量供审计。
        return logits, debug  # 返回词表分数与中间量。
model = TinyT5(len(vocabulary))  # 创建待训练的简化 T5。
parameter_count = sum(parameter.numel() for parameter in model.parameters())  # 统计手写模型参数量。
tied_storage = model.shared_embedding.weight.data_ptr()  # 记录实际共享输出矩阵的存储地址。
print(f"TinyT5 parameters={parameter_count} output_projection=shared_embedding_weight")  # 展示模型规模与确定性的权重共享证据。

TinyT5 parameters=21996 output_projection=shared_embedding_weight


## 4. 真实训练：teacher forcing、交叉熵与反向传播

六条样本组成完整 batch。每一步都重新计算 Encoder、causal Decoder、cross-attention 和 tied logits；loss 只统计非 padding 标签，然后执行真实 `backward()` 与 Adam 更新。

In [4]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.012)  # 创建同时更新 attention、FFN、bias 和共享 embedding 的优化器。
training_history = []  # 保存训练 loss、token accuracy 与梯度轨迹。
for step in range(601):  # 对受控六样本执行足够多次真实优化。
    model.train()  # 开启训练模式。
    logits, debug = model(source_ids, source_mask, decoder_input_ids, decoder_mask)  # 执行完整 teacher-forcing 前向传播。
    loss = torch.nn.functional.cross_entropy(logits.reshape(-1, len(vocabulary)), target_ids.reshape(-1), ignore_index=pad_id)  # 计算所有有效目标 token 的词表交叉熵。
    optimizer.zero_grad(set_to_none=True)  # 清除上一轮所有参数梯度。
    loss.backward()  # 从 tied logits 穿过 Decoder、cross-attention 和 Encoder 反向传播。
    gradient_norm = math.sqrt(sum(float((parameter.grad ** 2).sum().item()) for parameter in model.parameters() if parameter.grad is not None))  # 汇总当前模型梯度二范数。
    optimizer.step()  # 用 Adam 应用一次参数更新。
    if step % 150 == 0 or step == 600:  # 定期保存实际学习曲线。
        with torch.no_grad():  # 统计训练指标时关闭梯度。
            predictions = logits.argmax(dim=-1)  # 选择每个有效位置最高分 token。
            token_accuracy = ((predictions == target_ids) & target_mask).sum().item() / target_mask.sum().item()  # 计算 teacher-forcing token accuracy。
        training_history.append({"step": step, "loss": float(loss.item()), "token_accuracy": token_accuracy, "gradient_norm": gradient_norm})  # 保存可检查训练快照。
model.eval()  # 切换到确定性评估模式。
with torch.no_grad():  # 重新计算更新后最终输出。
    final_logits, final_debug = model(source_ids, source_mask, decoder_input_ids, decoder_mask)  # 获得最终 logits 和中间张量。
    final_predictions = final_logits.argmax(dim=-1)  # 取得 teacher-forcing 最大概率 token。
final_correct = final_predictions.eq(target_ids) & target_mask  # 标记最终有效位置命中。
t5_token_accuracy = final_correct.sum().item() / target_mask.sum().item()  # 计算和 baseline 同口径准确率。
future_mask = torch.triu(torch.ones(target_length, target_length, dtype=torch.bool), diagonal=1)  # 标记因果注意力矩阵的所有未来位置。
maximum_future_attention = final_debug["decoder_attention"][:, :, future_mask].max().item()  # 读取 Decoder 泄漏到未来的最大概率。
print("TinyT5训练轨迹=", training_history)  # 输出 loss、准确率和非零梯度演化。
print("Encoder memory shape=", tuple(final_debug["memory"].shape))  # 展示 Encoder memory 批张量。
print("样本1 Encoder head0 attention=", torch.round(final_debug["encoder_attention"][0, 0] * 1000) / 1000)  # 展示 source 双向注意力。
print("样本1 Decoder head0 causal attention=", torch.round(final_debug["decoder_attention"][0, 0] * 1000) / 1000)  # 展示下三角目标注意力。
print("样本1 Cross head0 attention=", torch.round(final_debug["cross_attention"][0, 0] * 1000) / 1000)  # 展示目标读取 source 的注意力。
print(f"最大未来attention={maximum_future_attention:.8f}")  # 验证 Decoder 没有偷看未来标签。

TinyT5训练轨迹= [{'step': 0, 'loss': 30.31858253479004, 'token_accuracy': 0.0, 'gradient_norm': 13.207894875374992}, {'step': 150, 'loss': 0.000246146199060604, 'token_accuracy': 1.0, 'gradient_norm': 0.0014789233438910314}, {'step': 300, 'loss': 0.00011558338883332908, 'token_accuracy': 1.0, 'gradient_norm': 0.0006431330297212051}, {'step': 450, 'loss': 6.839023990323767e-05, 'token_accuracy': 1.0, 'gradient_norm': 0.0003838309927612075}, {'step': 600, 'loss': 4.633923163055442e-05, 'token_accuracy': 1.0, 'gradient_norm': 0.00026092674936710605}]
Encoder memory shape= (6, 5, 32)
样本1 Encoder head0 attention= tensor([[0.1940, 0.4100, 0.2560, 0.1310, 0.0080],
        [0.0060, 0.0000, 0.9920, 0.0020, 0.0010],
        [0.1810, 0.0030, 0.0310, 0.2550, 0.5300],
        [0.0140, 0.9860, 0.0000, 0.0000, 0.0000],
        [0.0080, 0.0310, 0.1680, 0.0300, 0.7630]])
样本1 Decoder head0 causal attention= tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 1.0000, 0.0000, 0.0000, 0.0000],
 

## 5. 逐样本结果与结果解读：teacher forcing 与自回归生成

teacher forcing 检查每个目标位置的建模能力；greedy decoding 则只给 BOS，每次把自己刚预测的 token 追加回输入，更接近实际推理。两者差距可暴露 exposure bias。

In [5]:
def greedy_decode(single_source, single_mask, maximum_length):  # 对一条 source 执行真实自回归解码。
    generated = torch.tensor([[bos_id]], dtype=torch.long)  # 只用 BOS 初始化 Decoder。
    for _ in range(maximum_length):  # 逐 token 扩展直到 EOS 或长度上限。
        generated_mask = generated.ne(pad_id)  # 当前前缀中的所有 token 都有效。
        logits, _ = model(single_source, single_mask, generated, generated_mask)  # 对完整 source 和当前预测前缀前向传播。
        next_token = logits[:, -1].argmax(dim=-1, keepdim=True)  # 只读取最后位置的下一 token。
        generated = torch.cat([generated, next_token], dim=1)  # 把模型自己的预测追加回输入。
        if int(next_token.item()) == eos_id:  # 判断当前样本是否生成停止符。
            break  # 生成 EOS 后停止继续扩展。
    return generated[0, 1:]  # 去掉仅用于启动的 BOS。
greedy_outputs = [greedy_decode(source_ids[index:index + 1], source_mask[index:index + 1], target_length + 2) for index in range(len(raw_samples))]  # 逐条运行无 teacher token 的解码。
greedy_exact = []  # 保存逐样本去 padding 后 exact match。
print("sample  source                              target                              teacher                            greedy")  # 输出逐样本比较表头。
for index in range(len(raw_samples)):  # 逐条对比目标和两种输出。
    target_sequence = target_ids[index][target_mask[index]].tolist()  # 读取当前真实变长标签。
    greedy_sequence = greedy_outputs[index].tolist()  # 读取当前模型自回归结果。
    is_exact = greedy_sequence == target_sequence  # 判断完整 token 序列是否完全一致。
    greedy_exact.append(is_exact)  # 保存 exact-match 标志。
    teacher_accuracy = final_correct[index].sum().item() / target_mask[index].sum().item()  # 计算当前 teacher-forcing token accuracy。
    print(f"{index + 1:<6} {' '.join(corrupted_pairs[index][0]):<35} {decode_ids(target_ids[index]):<35} {decode_ids(final_predictions[index]):<35} {decode_ids(greedy_outputs[index])}")  # 输出可读 source、标签和预测。
    print(f"       teacher_token_acc={teacher_accuracy:.3f} greedy_exact={is_exact}")  # 输出当前样本量化结果。
greedy_exact_rate = sum(int(value) for value in greedy_exact) / len(greedy_exact)  # 计算六条序列 exact-match 比例。
first_position_probabilities = torch.softmax(final_logits[0, 0], dim=-1)  # 计算样本一首个目标 token 的词表概率。
top_probabilities, top_indices = first_position_probabilities.topk(5)  # 读取该位置最可能的五个 token。
top_candidates = [(id_to_token[int(index)], float(probability)) for probability, index in zip(top_probabilities, top_indices)]  # 转成可解释 token-probability 对。
print(f"同指标结果：baseline token_acc={baseline_accuracy:.3f}，TinyT5 token_acc={t5_token_accuracy:.3f}，greedy exact={greedy_exact_rate:.3f}")  # 汇总真实收益与解码差距。
print("样本1首位置top5=", top_candidates)  # 展示词表 logits 对实际选择的证据。

sample  source                              target                              teacher                            greedy
1      用户 <extra_id_0> 红色 <extra_id_1> 耳机  <extra_id_0> 搜索 <extra_id_1> 无线 <eos> <extra_id_0> 搜索 <extra_id_1> 无线 <eos> <extra_id_0> 搜索 <extra_id_1> 无线 <eos>
       teacher_token_acc=1.000 greedy_exact=True
2      客户 <extra_id_0> 门店 库存               <extra_id_0> 查询 北京 <eos>            <extra_id_0> 查询 北京 <eos> <eos>      <extra_id_0> 查询 北京 <eos>
       teacher_token_acc=1.000 greedy_exact=True
3      推荐 <extra_id_0> 游戏 <extra_id_1> 电脑  <extra_id_0> 轻薄 <extra_id_1> 笔记本 <eos> <extra_id_0> 轻薄 <extra_id_1> 笔记本 <eos> <extra_id_0> 轻薄 <extra_id_1> 笔记本 <eos>
       teacher_token_acc=1.000 greedy_exact=True
4      查找 <extra_id_0> 智能 手表               <extra_id_0> 防水 运动 <eos>            <extra_id_0> 防水 运动 <eos> <extra_id_0> <extra_id_0> 防水 运动 <eos>
       teacher_token_acc=1.000 greedy_exact=True
5      比较 <extra_id_0> 手机 <extra_id_1> 价格  <extra_id_0> 华为 <extra_id_1> 近期 <eos> <e

## 6. 失败案例与修正：两个 span 复用同一 sentinel

sentinel 是 source 与 target 之间的“片段地址”。若两个删除片段都叫 `<extra_id_0>`，重建器无法区分“搜索”和“无线”应填回哪个洞；为每个 span 使用唯一 sentinel 后映射才是一一对应。

In [6]:
def parse_target_chunks(tokens):  # 从目标序列解析 sentinel 到恢复内容的映射。
    chunks = {}  # 保存已经见过的片段地址。
    duplicate = False  # 初始化 sentinel 重复标志。
    current = None  # 保存当前正在收集的 sentinel。
    for token in tokens:  # 逐 token 扫描恢复目标。
        if token.startswith("<extra_id_"):  # 判断是否开始一个新片段。
            if token in chunks:  # 检查这个地址是否已被前一个片段占用。
                duplicate = True  # 标记地址冲突导致歧义。
            chunks[token] = []  # 为当前地址创建或覆盖内容列表。
            current = token  # 后续普通词归属于这个 sentinel。
        elif token != "<eos>" and current is not None:  # 忽略 EOS 并收集普通恢复词。
            chunks[current].append(token)  # 把词追加到当前片段。
    return chunks, duplicate  # 返回解析结果与地址冲突标志。
wrong_target = ["<extra_id_0>", "搜索", "<extra_id_0>", "无线", "<eos>"]  # 故意让两个不同 span 复用同一地址。
correct_target = ["<extra_id_0>", "搜索", "<extra_id_1>", "无线", "<eos>"]  # 为第二个 span 分配唯一地址。
wrong_chunks, wrong_duplicate = parse_target_chunks(wrong_target)  # 解析错误目标并复现覆盖。
correct_chunks, correct_duplicate = parse_target_chunks(correct_target)  # 解析修正目标并验证一一对应。
print(f"错误行为：target={' '.join(wrong_target)} duplicate={wrong_duplicate} parsed={wrong_chunks}，第一个片段被覆盖。")  # 展示真实地址冲突。
print(f"修正行为：target={' '.join(correct_target)} duplicate={correct_duplicate} parsed={correct_chunks}，两个洞可独立恢复。")  # 展示唯一 sentinel 修复结果。

错误行为：target=<extra_id_0> 搜索 <extra_id_0> 无线 <eos> duplicate=True parsed={'<extra_id_0>': ['无线']}，第一个片段被覆盖。
修正行为：target=<extra_id_0> 搜索 <extra_id_1> 无线 <eos> duplicate=False parsed={'<extra_id_0>': ['搜索'], '<extra_id_1>': ['无线']}，两个洞可独立恢复。


## 7. 生产边界

六条固定 span 只验证计算闭环。生产 T5 预训练需要在线随机噪声与噪声密度统计、sentinel 数量上限、文档级去重和污染审计、sequence packing、超长文截断、混合精度与分布式 checkpoint、held-out denoising/perplexity、下游任务迁移及 beam/search 解码监控。teacher-forcing 高分也不等于开放域生成可靠。

In [7]:
t5_diagnostics = {"samples": len(raw_samples), "source_shape": tuple(source_ids.shape), "target_tokens": int(target_mask.sum().item()), "baseline_token_accuracy": baseline_accuracy, "t5_token_accuracy": t5_token_accuracy, "greedy_exact_rate": greedy_exact_rate, "initial_loss": training_history[0]["loss"], "final_loss": training_history[-1]["loss"], "maximum_future_attention": maximum_future_attention, "wrong_duplicate": wrong_duplicate, "correct_duplicate": correct_duplicate}  # 汇总数据、训练、解码、mask 和失败修正指标。
print("生产监控快照：", t5_diagnostics)  # 输出序列去噪系统应持续观察的核心信号。

生产监控快照： {'samples': 6, 'source_shape': (6, 5), 'target_tokens': 27, 'baseline_token_accuracy': 0.5925925925925926, 't5_token_accuracy': 1.0, 'greedy_exact_rate': 1.0, 'initial_loss': 30.31858253479004, 'final_loss': 4.633923163055442e-05, 'maximum_future_attention': 0.0, 'wrong_duplicate': True, 'correct_duplicate': False}


## 8. 最小回归测试

最后一格只保护真实样本规模、训练收益、因果 mask、共享权重、自回归结果和 sentinel 修正；教学证据已经在前面完整输出。

In [8]:
assert len(raw_samples) >= 5 and all(len(pair[0]) >= 3 and len(pair[1]) >= 3 for pair in corrupted_pairs)  # 保证案例不是单条占位输入。
assert training_history[-1]["loss"] < training_history[0]["loss"] and all(row["gradient_norm"] > 0.0 for row in training_history)  # 保证完整 Encoder-Decoder 真实 backward 学习。
assert t5_token_accuracy > baseline_accuracy and t5_token_accuracy >= 0.90  # 保证同口径 token accuracy 显著优于位置众数。
assert greedy_exact_rate >= 0.50  # 保证至少一半样本可在没有 teacher token 时完整恢复。
assert maximum_future_attention < 1.0e-6 and model.shared_embedding.weight.data_ptr() == tied_storage  # 保证因果约束与输出权重绑定有效。
assert wrong_duplicate and not correct_duplicate and len(correct_chunks) == 2  # 保证 sentinel 冲突可复现且唯一编号真正修复。